Before running the file Upload all your data set on your goole drive in a zip format

In [2]:
#To get the average frame count 
import json
import glob
import numpy as np
import cv2
import copy
#change the path accordingly
video_files =  glob.glob('./real_yt/*.mp4')
frame_count = []
for video_file in video_files:
  cap = cv2.VideoCapture(video_file)
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))<150):
    video_files.remove(video_file)
    continue
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("frames" , frame_count)
print("Total number of videos: " , len(frame_count))
print('Average frame per video:',np.mean(frame_count))

frames [564, 355, 407, 651, 444, 626, 535, 637, 390, 493, 1016, 407, 323, 304, 413, 333, 686, 1228, 594, 427, 975, 485, 452, 444, 880, 801, 390, 319, 357, 558, 302, 311, 448, 578, 455, 420, 299, 336, 616, 346, 487, 620, 426, 319, 338, 550, 374, 508, 829, 512, 649, 434, 690, 557, 318, 345, 686, 423, 431, 470, 588, 383, 379, 757, 727, 436, 336, 476, 337, 364, 312, 718, 496, 600, 475, 306, 454, 960, 331, 569, 553, 328, 460, 383, 520, 305, 414, 332, 424, 486, 351, 411, 569, 567, 320, 468, 827, 499, 532, 313, 525, 355, 465, 460, 360, 335, 293, 761, 333, 478, 419, 297, 423, 518, 962, 350, 382, 502, 590, 692, 651, 294, 325, 337, 441, 455, 328, 359, 541, 357, 714, 303, 528, 465, 372, 372, 320, 488, 353, 565, 382, 401, 310, 322, 344, 348, 330, 434, 403, 533, 412, 628, 1003, 313, 338, 1205, 875, 372, 650, 360, 631, 316, 518, 405, 761, 528, 579, 1496, 449, 341, 593, 684, 423, 1814, 433, 301, 331, 1254, 489, 732, 764, 370, 667, 364, 311, 556, 1001, 484, 315, 370, 335, 336, 809, 292, 294, 350, 718,

In [3]:
import torch
print(f"MPS (Metal) supported: {torch.backends.mps.is_available()}")

MPS (Metal) supported: True


In [5]:
import os
import glob
import cv2
import numpy as np
import face_recognition
from tqdm.auto import tqdm  # Fixed tqdm import
import warnings

# Suppress unnecessary warnings
warnings.filterwarnings('ignore')

def frame_extract(path, max_frames=150):
    """Improved frame extraction with early termination"""
    vidObj = cv2.VideoCapture(path)
    count = 0
    while count < max_frames:
        success, image = vidObj.read()
        if not success:
            break
        yield image
        count += 1
    vidObj.release()

def create_face_videos(path_list, out_dir, batch_size=4):
    """Optimized face video creation with proper video codec handling"""
    os.makedirs(out_dir, exist_ok=True)
    
    # Count existing videos more efficiently
    existing_files = set(glob.glob(os.path.join(out_dir, '*.mp4')))
    print(f"Found {len(existing_files)} already processed videos")
    
    for path in tqdm(path_list, desc="Processing videos"):
        out_path = os.path.join(out_dir, os.path.basename(path))
        
        # Skip if already processed
        if out_path in existing_files:
            continue
            
        # Use proper MP4 codec for macOS
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Changed from MJPG to mp4v
        out = cv2.VideoWriter(
            out_path,
            fourcc,
            30,
            (112, 112),
            isColor=True
        )
        
        frames = []
        try:
            for frame in frame_extract(path):
                frames.append(frame)
                
                if len(frames) == batch_size:
                    # Batch processing with face detection
                    face_batches = face_recognition.batch_face_locations(
                        frames,
                        number_of_times_to_upsample=0,
                        batch_size=batch_size
                    )
                    
                    for i, face_locations in enumerate(face_batches):
                        if face_locations:
                            top, right, bottom, left = face_locations[0]
                            try:
                                face_img = frames[i][top:bottom, left:right]
                                resized = cv2.resize(face_img, (112, 112))
                                out.write(resized)
                            except Exception as e:
                                continue
                                
                    frames = []
        finally:
            out.release()
            
        print(f"Completed: {out_path}")


In [6]:
video_files = glob.glob('./fake_yt/*.mp4')  # Adjust path as needed

In [7]:

create_face_videos(video_files, './fake_yt_face_videos/')

Found 0 already processed videos


Processing videos:   0%|          | 1/1000 [01:02<17:19:13, 62.42s/it]

Completed: ./fake_yt_face_videos/475_265.mp4


Processing videos:   0%|          | 2/1000 [01:24<10:41:40, 38.58s/it]

Completed: ./fake_yt_face_videos/366_473.mp4


...

Completed: ./fake_yt_face_videos/268_269.mp4


Processing videos:   2%|▏         | 19/1000 [17:03<10:36:09, 38.91s/it]

Completed: ./fake_yt_face_videos/503_756.mp4


Processing videos:   2%|▏         | 19/1000 [18:01<15:30:42, 56.92s/it]


KeyboardInterrupt: 